middleware

Middleware provides a way to mroe tightly control whats happen inside agent. Middlewarre ois useful for the following:

1) Tracking agent behavoir wiht logging , analytics , and debugging
2) Transforming prompts , tools selection and outptu formatting
3) adding retries, fallback, and early termination logic
4) Applying rate limits , guadrails, and PII detection



## Built-in Middleware in LangChain

### 1. SummarizationMiddleware
- Automatically summarizes conversation history.
- Prevents context window/token limit issues.
- Keeps only the most important information.
- Useful for long-running chat agents.

### 2. HumanInTheLoopMiddleware
- Pauses execution for human approval before sensitive tool calls.
- Allows **Approve**, **Edit**, or **Reject** actions.
- Used for emails, payments, database writes, etc.

### 3. ModelCallLimitMiddleware
- Limits the number of LLM calls.
- Prevents infinite loops.
- Helps control API costs.

### 4. ToolCallLimitMiddleware
- Restricts how many times tools can be executed.
- Prevents excessive or repeated tool usage.
- Improves safety and cost efficiency.

### 5. ModelFallbackMiddleware
- Switches to a backup model if the primary model fails.
- Increases reliability and availability.
- Useful in production systems.

### 6. PIIMiddleware
- Detects Personally Identifiable Information (PII).
- Can redact, mask, hash, or block sensitive data.
- Protects user privacy and ensures compliance.

### 7. TodoListMiddleware
- Gives agents task-planning capabilities.
- Creates and manages to-do lists.
- Useful for multi-step and long-running tasks.

### 8. LLMToolSelectorMiddleware
- Uses an LLM to choose the most relevant tools.
- Avoids unnecessary tool calls.
- Improves agent efficiency.

### 9. ToolRetryMiddleware
- Automatically retries failed tool executions.
- Handles temporary errors like network failures.
- Improves robustness.

### 10. ModelRetryMiddleware
- Retries failed LLM API calls.
- Supports configurable retry logic and backoff.
- Useful for transient model/API failures.

### 11. LLMToolEmulator
- Simulates tool execution using an LLM.
- Useful for testing or when real tools are unavailable.
- Reduces dependency on external services.

### 12. ContextEditingMiddleware
- Edits or prunes conversation context before sending it to the model.
- Removes unnecessary information.
- Helps reduce token usage.

### 13. FilesystemMiddleware
- Controls how agents access files and directories.
- Restricts file operations for security.
- Useful for file-based AI assistants.

### 14. ShellTool Middleware
- Controls execution of shell/terminal commands.
- Adds safety checks before running commands.
- Useful for coding and DevOps agents.

### 15. FileSearch Middleware
- Enables controlled searching across files.
- Helps agents retrieve relevant documents efficiently.
- Commonly used in RAG and coding assistants.

In [1]:

import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

model_name="google_genai:gemini-2.5-flash"
model=init_chat_model(model_name)


Summarization

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1) Long-running conversations that exceed context windows.
2) Multi-turn dialogues with extensive history.
3) Applications where preserving full conversation context matters.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


Message Summarization

In [ ]:
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("messages",8),
        keep=("messages",4)   
    )]
)

In [22]:
#Run wih thread id
config={"configurable":{"thread_id":"test-1"}}

In [23]:
questions=[
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4"

]

In [24]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='6f76a762-6951-4da9-84cf-677cbafddd08'), AIMessage(content='2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4be1-32a6-7373-a1b3-32e746667671-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='6f76a762-6951-4da9-84cf-677cbafddd08'), AIMessage(content='2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4be1-32a6-7373-a1b3-32e746667671-0', tool_calls=[], inval

TOken Size 

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

In [26]:
@tool
def search_hotels(city:str) -> str:
    """Search the hotels - return long response to use more tokens"""
    return f"""Hotels in {city}:
    1.Grand hotel - 5 star , 350/night, spa pool gym
    2. CIty inn 4 sstar 100 night businees center
    3. budget stay - 3 star , 75/night , free wifi """

In [35]:
agent=create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="google_genai:gemini-3.1-flash-lite",
        trigger=("tokens",550),
        keep=("tokens",200)   
    )]
)
#Run wih thread id
config={"configurable":{"thread_id":"test-1"}}


# token counter
def count_tokens(messages):
    total_char=sum(len(str(m.content))for m in messages)
    return total_char//4
    

In [ ]:
cities =["Paris","London","Tokyo","New York","Singapore","Dubai"]

for city in cities:
    response=agent.invoke(
        {
            "messages":[HumanMessage(content=f"Find the hotels in {city}")]
        },
        config=config
    )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens ,{len(response["messages"])}messages")
    print(f"{(response["messages"])}")

[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is researching and comparing hotel options across multiple international cities (Paris, London, Tokyo, New York, Singapore, and Dubai) based on standardized pricing and amenity tiers to facilitate a comparative analysis.\n\n## SUMMARY\n\nThe assistant has established a standardized hotel categorization system applied consistently across six cities. Each city features the same three tier options: \n1. **Grand Hotel**: 5-star, $350/night (includes spa, pool, gym).\n2. **City Inn**: 4-star, $100/night (includes business center).\n3. **Budget Stay**: 3-star, $75/night (includes free Wi-Fi).\n\nThis uniform structure allows for direct price and amenity comparison across all documented locations. Dubai was the most recent city added to this dataset.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\nAwait further user instructions, which may include searching for additional cities, refining searc

KeyboardInterrupt: 

Fraction

F

In [ ]:
agent_fract=create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="google_genai:gemini-3.1-flash-lite",
        trigger=("fraction",0.005), #0.5% of total tokens of the models used
        keep=("fraction",0.002)   #0.2% of the total tokens of the models used
    )]
)
#Run wih thread id
config={"configurable":{"thread_id":"test-1"}}


# token counter
def count_tokens(messages):
    total_char=sum(len(str(m.content))for m in messages)
    return total_char//4

In [42]:
cities =["Paris","London","Tokyo","New York","Singapore","Dubai"]

for city in cities:
    response=agent_fract.invoke(
        {
            "messages":[HumanMessage(content=f"Find the hotels in {city}")]
        },
        config=config
    )
    tokens=count_tokens(response["messages"])
    fraction=tokens/1000000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}),{len(response["messages"])}messages")
    print(f"{(response["messages"])}")

Paris: ~1022 tokens (0.1022%),28messages
[HumanMessage(content='Find the hotels in Paris', additional_kwargs={}, response_metadata={}, id='75b620cb-fa08-4e3c-a92d-ff11a65ab96f'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'2egNSkeQ': 'EjQKMgERTTIPexg8Twhro6/Lr6PEbgvT2WRlS9+u5fx6yN75Lm6HNkniSDYwk3cx5kMGqzas'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4fb2-5892-79f0-98da-f14e7131da14-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '2egNSkeQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 16, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n    1.Grand hotel - 5 star , 350/night, spa pool gym\n    2. CIty inn 4 sstar 100 

Human in the loop

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

1 High-stakes operations requiring human approval (e.g. database writes, financial transactions).

2 Compliance workflows where human oversight is mandatory.

3 Long-running conversations where human feedback guides the agent.

In [54]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email(email:str)-> str:
    """MOck func to read"""
    return f"Email content for the email{email}"

def send_email(email:str,sub:str,body:str)->str:
    """MOck func to send email"""
    return f"The email has been sent to {email} with the sub: {sub}"




In [57]:
agent_approv=create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email,send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email":False
            }
        )
        ]
)

In [58]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='8124deda-9b0b-41cf-a62d-068c143911bb'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'8c7AgFWZ': 'EjQKMgERTTIPkN+JV4R9oHA1Y239/gUnMnoIkSILm8Qczmm7O/ESTVWBkfzN1wibs5/huLYw'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4fd2-f0e7-7d50-9856-18021ec12bdf-0', tool_calls=[{'name': 'send_email', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': '8c7AgFWZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 35, 'total_tokens': 104, 'input_token_details': {'cache_rea

In [63]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email,send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email":False,

            }
        )
    ]
)

In [65]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='8b2eff28-a5e9-472f-bf04-daa926b7385e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available functions. There's a send_email function that requires email, sub, and body. The parameters match what the user provided. I need to make sure all required fields are included. The email is there, the subject is 'Hello', and the body is 'How are you?'. All required parameters are present. I'll call the send_email function with these arguments.\n", 'tool_calls': [{'id': '4r8tg05jw', 'function': {'arguments': '{"body":"How are you?","email":"john@test.com","sub":"Hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 243, 

In [66]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f" Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
 Result: The email has been successfully sent to **john@test.com** with the subject **"Hello"** and the message **"How are you?"**. Let me know if you need anything else!


similarly we can do it for reject alsp and edit

EDIT

In [68]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)


In [69]:

config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [70]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='770a41d9-2438-4d7b-bcec-2a989963686b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required, and the user provided all. So I should call send_email_tool with those arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': 'tybt1wgra', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject

In [71]:

# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f" Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
 Result: The email was sent to **correct@email.com** with the subject **"Corrected Subject"** instead of the requested details. This suggests the system automatically modified the recipient and subject before sending. Would you like to:

1. Confirm if these changes were intentional.
2. Send the email with the original details as specified.
3. Investigate why the system altered the parameters?

Let me know how you'd like to proceed.
